# Frame Captioning Extractor v2.2

Optimized for Kaggle **2× NVIDIA T4**:

- true batched benchmark instead of one-image-at-a-time inference;
- replicated inference: one quantized model copy per GPU;
- bounded CPU preprocessing workers per GPU;
- periodic CUDA metrics instead of synchronizing after every production batch;
- Qwen3-VL-4B 4-bit defaults tuned for 2×T4 throughput.


## 1. Install Dependencies Safely


In [ ]:
from importlib import metadata
from pathlib import Path
import subprocess
import sys

REQUIRED_PACKAGES = [
    "transformers==4.57.6",
    "qwen-vl-utils==0.0.14",
    "accelerate>=0.26.0,<2.0",
    "bitsandbytes>=0.48.2,<0.50",
    "sentencepiece>=0.2.0,<0.3",
    "google-cloud-storage>=2.10.0,<4.0.0",
    "tqdm>=4.66.0,<5.0.0",
]

# Preserve packages managed by the notebook runtime. Their exact installed
# versions are written as constraints, so pip cannot replace them indirectly.
PROTECTED_PACKAGES = [
    "numpy",
    "pandas",
    "Pillow",
    "torch",
    "torchvision",
    "torchaudio",
    "jupyter-server",
    "decorator",
    "gym",
    "numba",
]

constraints = []
for package_name in PROTECTED_PACKAGES:
    try:
        constraints.append(f"{package_name}=={metadata.version(package_name)}")
    except metadata.PackageNotFoundError:
        pass

constraints_path = Path("/tmp/captioning-runtime-constraints.txt")
constraints_path.write_text("\n".join(constraints) + "\n", encoding="utf-8")

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--disable-pip-version-check",
    "--upgrade-strategy",
    "only-if-needed",
    "--constraint",
    str(constraints_path),
    *REQUIRED_PACKAGES,
]
subprocess.check_call(command)

# Validate package metadata without importing/reloading large libraries.
expected_exact_versions = {
    "transformers": "4.57.6",
    "qwen-vl-utils": "0.0.14",
}
for package_name, expected_version in expected_exact_versions.items():
    installed_version = metadata.version(package_name)
    if installed_version != expected_version:
        raise RuntimeError(
            f"{package_name} version mismatch: "
            f"expected {expected_version}, found {installed_version}"
        )

print("Dependency installation completed without replacing runtime core packages.")
print("Protected runtime versions:")
for requirement in constraints:
    print(" -", requirement)
print("Model stack:")
for package_name in ["transformers", "qwen-vl-utils", "accelerate", "bitsandbytes"]:
    print(f" - {package_name}=={metadata.version(package_name)}")


## 2. Parameters

### Cách cấu hình Kaggle Secrets

Trong Kaggle Notebook, mở **Add-ons → Secrets** và tạo đúng hai secret:

- `GCS_BUCKET`: tên bucket, ví dụ `aic_ai_2026` hoặc `gs://aic_ai_2026`.
- `GCS_CREDENTIALS_JSON`: toàn bộ nội dung JSON của Google Cloud service account.

Notebook không ghi cứng bucket hoặc credential. Khi chạy, `UserSecretsClient` đọc hai secret trên và tạo `google.cloud.storage.Client`.

### Cách model được load

Notebook dùng trực tiếp:

```python
ModelClass.from_pretrained("organization/model-name")
AutoProcessor.from_pretrained("organization/model-name")
```

Không có cell `snapshot_download`, không copy checkpoint vào `/kaggle/working`, và không upload weight vào GCS.

- `HF_CACHE_DIR = ""`: dùng cache mặc định do Hugging Face/Transformers quản lý, không dùng thư mục Kaggle Output.
- `TRANSFORMERS_LOCAL_FILES_ONLY = False`: cho phép Transformers lấy checkpoint từ Hugging Face Hub khi cache chưa có.
- Download checkpoint chỉ bắt đầu khi cell Benchmark, Demo hoặc Full Run thực sự gọi model.
- Thanh tiến trình tải shard của Hugging Face được bật để bạn theo dõi.
- Benchmark mặc định chỉ có `ACTIVE_MODEL_KEY`, tránh tải cả ba model trong một session.

### Cách chạy trên Kaggle

1. Bật **GPU** và **Internet** trong Notebook Settings.
2. Tạo hai Kaggle Secrets và cấp quyền cho notebook.
3. Chỉnh `ACTIVE_MODEL_KEY` trong Parameters.
4. Chạy lần lượt: **Install Dependencies → Parameters → Shared Helpers → Captioning Model Logic**.
5. Chạy **Dry Run**. Cell này không tải ảnh và không load model.
6. Chạy **Benchmark 5 frames**. Notebook chỉ lấy đúng 5 frame và chạy model đã chọn.
7. Kiểm tra bảng, ảnh, caption, thời gian và VRAM được hiển thị.
8. Chạy **Demo Run** nếu benchmark ổn.
9. Chỉ chạy **Full Run** sau khi đặt `CONFIRM_FULL_RUN = "RUN_FULL_DATASET"`.

### Benchmark nhiều model

Mặc định:

```python
BENCHMARK_MODEL_KEYS = [ACTIVE_MODEL_KEY]
```

Để so sánh thêm model, thêm model key thủ công. Mỗi model được load tuần tự và giải phóng khỏi VRAM trước khi model tiếp theo được load. Tuy nhiên checkpoint vẫn có thể chiếm disk cache, nên nên chạy từng model ở các Kaggle session riêng khi disk hạn chế.

### GCS output layout

```text
features/extractors/
└── dataset={DATASET_ID}/
    └── batch={BATCH_ID}/
        └── frame_profile={PROFILE_VERSION}/
            └── extractor=captioning/
                └── extractor_version={EXTRACTOR_VERSION}/
                    └── model={MODEL_KEY}/
                        ├── data/
                        │   ├── annotations.jsonl
                        │   ├── manifest.json
                        │   └── _SUCCESS
                        ├── runs/
                        │   └── run_id={RUN_ID}/
                        │       ├── annotations.jsonl
                        │       ├── errors.jsonl
                        │       ├── metrics.csv
                        │       ├── summary.json
                        │       ├── config.json
                        │       ├── run.log
                        │       └── _SUCCESS
                        └── latest.json
```

- `data/` là canonical feature cho downstream.
- `runs/` giữ lineage, metric và lỗi của từng lần chạy.
- `latest.json` trỏ đến run đã publish canonical gần nhất.
- `extractor_version` là phiên bản pipeline; `model` là checkpoint/model key.


In [ ]:
from types import SimpleNamespace

# ---------------------------------------------------------------------------
# GCS and dataset identity
# ---------------------------------------------------------------------------
# Values are read from Kaggle Secrets. Do not paste credentials into this notebook.
GCS_BUCKET_SECRET_NAME = "GCS_BUCKET"
GCS_CREDENTIALS_JSON_SECRET_NAME = "GCS_CREDENTIALS_JSON"
REQUIRE_KAGGLE_GCS_SECRETS = True

DATASET_ID = "ai_challenge_2025"
PROFILE_VERSION = "autoshot_v1"
BATCHES = [
    "L21",
    # "L22",
    # "L23",
    # "L24",
    # "L26",
    # "L27",
    # "L28",
    # "L29",
    # "L30",
]

# Frame extraction layout on GCS.
KEYFRAMES_PREFIX = "processed/keyframes"
MANIFESTS_PREFIX = "processed/keyframes_manifests"
INPUT_MANIFEST_URI = ""  # Optional explicit gs://.../shot_segments.csv or frames_manifest.jsonl.

# ---------------------------------------------------------------------------
# Caption prompts
# ---------------------------------------------------------------------------
CAPTION_PROMPT_VI = """
Bạn là hệ thống tạo caption cho ảnh/keyframe video.

Nhiệm vụ:
- Mô tả nội dung chính của ảnh bằng tiếng Việt.
- Tập trung vào người, hành động, vật thể, bối cảnh, địa điểm và sự kiện.
- Nếu có chữ, logo, biển báo hoặc phụ đề quan trọng thì ghi lại phần nhìn thấy rõ.
- Không suy đoán thông tin không chắc chắn.
- Không mô tả quá dài.

Định dạng trả về:
Một đoạn văn tiếng Việt ngắn gọn, từ 1 đến 3 câu.
""".strip()

CAPTION_PROMPT_EN = """
You are an image and video keyframe captioning system.

Task:
- Describe the main content of the image in English.
- Focus on people, actions, objects, context, locations, and events.
- If the image contains visible text, logos, signs, or subtitles, include the important text that can be clearly read.
- Do not infer or speculate about uncertain information.
- Keep the description concise.

Output format:
Return one concise paragraph in English, consisting of 1–3 sentences.
""".strip()

# Giữ prompt gốc của file get-features cho BLIP-2.
BLIP_PROMPT = "a photo of"

# ---------------------------------------------------------------------------
# Model registry
# Add another model with an existing backend by copying one entry below.
# Supported backends: qwen25_vl, blip2, qwen3_vl
# ---------------------------------------------------------------------------
MODEL_REGISTRY = {
    "qwen25_vl_3b": {
        "backend": "qwen25_vl",
        "hf_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "prompt": CAPTION_PROMPT_EN,
        "batch_size": 8,
        "max_new_tokens": 200,
        "quantization": "none",
        "max_pixels": 1024 * 1024,
    },

    "blip2_opt_2_7b": {
        "backend": "blip2",
        "hf_id": "Salesforce/blip2-opt-2.7b",
        "prompt": BLIP_PROMPT,
        "batch_size": 8,
        "max_new_tokens": 200,
        "min_new_tokens": 10,
        "num_beams": 1,
        "repetition_penalty": 1.2,
        "length_penalty": 1.1,
        "quantization": "none",
    },

    "qwen3_vl_4b": {
        "backend": "qwen3_vl",
        "hf_id": "Qwen/Qwen3-VL-4B-Instruct",
        "prompt": CAPTION_PROMPT_EN,
        "batch_size": 4,
        "max_new_tokens": 64,
        "quantization": "4bit",
        "max_pixels": 640 * 640,
        "preprocess_workers": 2,
    },

    "qwen3_vl_8b": {
        "backend": "qwen3_vl",
        "hf_id": "Qwen/Qwen3-VL-8B-Instruct",
        "prompt": CAPTION_PROMPT_EN,
        "batch_size": 2,
        "max_new_tokens": 64,
        "quantization": "4bit",  # Set "none" only on a larger-VRAM GPU.
        "max_pixels": 640 * 640,
        "preprocess_workers": 2,
    },
}

ACTIVE_MODEL_KEY = "qwen3_vl_4b"

# Benchmark only loads models explicitly listed here.
# Keep the default as one model to avoid filling the Kaggle disk cache.
BENCHMARK_MODEL_KEYS = [ACTIVE_MODEL_KEY]

# ---------------------------------------------------------------------------
# Extractor output layout on GCS
# Pipeline version and model checkpoint are separate partitions.
# ---------------------------------------------------------------------------
OUTPUT_PREFIX = "features/extractors"
EXTRACTOR_VERSION = "fe-captioning-v2.2"
ANNOTATION_VERSION = "fe-captioning-v2.2"
MODEL_VERSION = MODEL_REGISTRY[ACTIVE_MODEL_KEY]["hf_id"]

CANONICAL_ANNOTATIONS_FILENAME = "annotations.jsonl"
CANONICAL_MANIFEST_FILENAME = "manifest.json"
LATEST_POINTER_FILENAME = "latest.json"

# Full runs publish canonical data. Demo runs remain under runs/ by default.
PUBLISH_CANONICAL_ON_FULL = True
PUBLISH_CANONICAL_ON_DEMO = False
MERGE_WITH_EXISTING_CANONICAL = True
WRITE_LATEST_POINTER = True

# Kaggle local runtime.
RUN_ROOT = "/kaggle/working/feature_extractor_runs"
SCRATCH_DIR = "/kaggle/working/feature_extractor_scratch"
# Empty means the standard Hugging Face cache (normally ~/.cache/huggingface).
# Do not point this at /kaggle/working unless you intentionally want checkpoint
# files to appear in Kaggle Output.
HF_CACHE_DIR = ""
TRANSFORMERS_LOCAL_FILES_ONLY = False
SHOW_HF_DOWNLOAD_PROGRESS = True
CLEANUP_LOCAL_FRAMES_AFTER_RUN = True

# ---------------------------------------------------------------------------
# Execution controls
# ---------------------------------------------------------------------------
DRY_RUN_MAX_FRAMES = 20

# Use enough frames to measure steady-state batch throughput.
BENCHMARK_BATCHES = ["L21"]
BENCHMARK_FRAME_LIMIT = 32
BENCHMARK_WARMUP = True
BENCHMARK_SAVE_CSV = True
BENCHMARK_SHOW_IMAGES = False
BENCHMARK_IMAGE_WIDTH = 420

DEMO_BATCHES = ["L21"]
DEMO_MAX_FRAMES = 32
FULL_MAX_FRAMES = None
CONFIRM_FULL_RUN = ""  # Set to RUN_FULL_DATASET before full run.

# Resume and failure behavior.
UPLOAD_TO_GCS = True
UPLOAD_RUN_ARTIFACTS = True
SKIP_EXISTING = True
OVERWRITE = False
RESUME_ANNOTATIONS_URI = ""
FAIL_FAST = False

# Parallelism and progress.
DOWNLOAD_WORKERS = 8
UPLOAD_WORKERS = 8

# Global outer batch. In replicated mode with 2 GPUs and per-GPU batch=4,
# one outer batch of 8 is split into 4 images for GPU 0 and 4 for GPU 1.
PIPELINE_BATCH_SIZE = 8

# CUDA memory collection synchronizes devices; sample it periodically.
METRICS_EVERY_N_BATCHES = 10

LOG_EVERY_N_FRAMES = 64
USE_TQDM = True

# Model runtime.
DEVICE = "auto"  # "auto", "cuda", or "cpu"

# GPU execution modes:
# - "replicated": one independent model copy per GPU; best throughput for 4B 4-bit.
# - "single": one model on the first GPU in GPU_IDS.
# - "sharded": one model split by device_map="auto"; use only when it cannot fit.
GPU_EXECUTION_MODE = "replicated"
GPU_IDS = [0, 1]

USE_FLASH_ATTENTION_2 = False
TRUST_REMOTE_CODE = False
LOW_CPU_MEM_USAGE = True

cfg = SimpleNamespace(**{
    name: value
    for name, value in globals().copy().items()
    if name.isupper() and not name.startswith("_")
})
cfg.EXTRACTOR_NAME = "captioning"

if cfg.ACTIVE_MODEL_KEY not in cfg.MODEL_REGISTRY:
    raise KeyError(f"Unknown ACTIVE_MODEL_KEY={cfg.ACTIVE_MODEL_KEY}")
unknown_benchmark_models = [
    key for key in cfg.BENCHMARK_MODEL_KEYS
    if key not in cfg.MODEL_REGISTRY
]
if unknown_benchmark_models:
    raise KeyError(f"Unknown BENCHMARK_MODEL_KEYS={unknown_benchmark_models}")

print("Parameters loaded for Frame Captioning Extractor v2.2.")
print("GCS bucket secret:", cfg.GCS_BUCKET_SECRET_NAME)
print("GCS credential secret:", cfg.GCS_CREDENTIALS_JSON_SECRET_NAME)
print("Active model:", cfg.ACTIVE_MODEL_KEY, cfg.MODEL_REGISTRY[cfg.ACTIVE_MODEL_KEY]["hf_id"])
print("Benchmark models:", cfg.BENCHMARK_MODEL_KEYS)
print("Batches:", cfg.BATCHES, "Demo:", cfg.DEMO_BATCHES, "Upload:", cfg.UPLOAD_TO_GCS)
print("Canonical publish: full=", cfg.PUBLISH_CANONICAL_ON_FULL, "demo=", cfg.PUBLISH_CANONICAL_ON_DEMO)

print("Benchmark frame limit:", cfg.BENCHMARK_FRAME_LIMIT)
print("HF cache:", cfg.HF_CACHE_DIR or "Transformers default cache")

print("GPU execution:", cfg.GPU_EXECUTION_MODE, "GPU_IDS=", cfg.GPU_IDS)
print("Pipeline batch:", cfg.PIPELINE_BATCH_SIZE, "Metrics every:", cfg.METRICS_EVERY_N_BATCHES, "batches")


In [ ]:
import os

print("Active model key:", cfg.ACTIVE_MODEL_KEY)
print("Model:", cfg.MODEL_REGISTRY[cfg.ACTIVE_MODEL_KEY]["hf_id"])
print("Quantization:", cfg.MODEL_REGISTRY[cfg.ACTIVE_MODEL_KEY]["quantization"])
print("Per-GPU model batch:", cfg.MODEL_REGISTRY[cfg.ACTIVE_MODEL_KEY]["batch_size"])
print("Pipeline/global batch:", cfg.PIPELINE_BATCH_SIZE)
print("Logical CPU count:", os.cpu_count())

try:
    import torch

    print("CUDA available:", torch.cuda.is_available())
    print("Visible GPU count:", torch.cuda.device_count())
    for gpu_id in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(gpu_id)
        print(
            f"GPU {gpu_id}: {props.name} | "
            f"{props.total_memory / 1024**3:.1f} GiB"
        )
except Exception as exc:
    print("Hardware inspection failed:", exc)


### v2.2 execution strategy

The default configuration is designed for Kaggle's two T4 GPUs:

```text
outer batch of 8 frames
├── GPU 0: Qwen3-VL-4B 4-bit, batch 4
└── GPU 1: Qwen3-VL-4B 4-bit, batch 4
```

`GPU_EXECUTION_MODE = "replicated"` loads one independent model copy on each
GPU. This is different from `device_map="auto"`, which splits one model across
GPUs and often leaves one T4 waiting during autoregressive generation.

Start with the 32-frame benchmark. Increase the model's per-GPU `batch_size`
from 4 to 6 only after confirming that each GPU stays below its VRAM limit.


## 3. Shared GCS, Manifest, Run Helpers

- Đọc `GCS_BUCKET` và `GCS_CREDENTIALS_JSON` từ Kaggle Secrets.
- Đọc manifest và tải keyframe từ GCS.
- Tạo partition `model={MODEL_KEY}`.
- Lưu artifact từng lần chạy trong `runs/run_id=.../`.
- Merge + deduplicate canonical annotations trong `data/`.
- Ghi `_SUCCESS` cuối cùng và cập nhật `latest.json`.


In [ ]:
from __future__ import annotations

import csv
import json
import logging
import os
import re
import shutil
import time
import uuid
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
@dataclass
class RunLayout:
    """Local paths and partitioned GCS paths for one extractor run."""
    run_id: str
    batch_id: str
    model_key: str
    run_dir: Path
    frames_dir: Path
    artifacts_dir: Path
    model_base_prefix: str
    data_prefix: str
    runs_prefix: str
    output_prefix: str  # Exact runs/run_id=.../ prefix.
    latest_pointer_blob: str

    # Path to the annotations for this extractor version.
    annotations_path: Path
    errors_path: Path
    metrics_path: Path
    summary_path: Path
    config_path: Path
    publish_manifest_path: Path
    published_annotations_path: Path
    log_path: Path

### Helper function

In [ ]:
def utc_now() -> str:
    """Return an ISO-8601 UTC timestamp."""
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def cfg_value(config: Any, name: str, default: Any = None) -> Any:
    """Read a value from a SimpleNamespace-like config object."""
    return getattr(config, name, default)


def normalize_prefix(value: str) -> str:
    """Normalize a GCS object prefix without leading/trailing slashes."""
    return str(value or "").strip().strip("/")


def make_run_id(kind: str) -> str:
    """Create a unique, sortable run identifier."""
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{kind}_{stamp}_{uuid.uuid4().hex[:8]}"

### Connect Google Cloud 

input: GCS_CREDENTIALS_JSON

In [ ]:
def read_kaggle_secret(secret_name: str, required: bool = False) -> str:
    """Read one Kaggle Secret without logging its value."""
    if not secret_name:
        if required:
            raise RuntimeError("A required Kaggle Secret name is empty.")
        return ""
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(secret_name)
    except Exception as exc:
        if required:
            raise RuntimeError(
                f"Cannot read Kaggle Secret {secret_name!r}. "
                "Create it in Add-ons -> Secrets and grant this notebook access."
            ) from exc
        return ""
    value = str(value or "").strip()
    if required and not value:
        raise RuntimeError(f"Kaggle Secret {secret_name!r} is empty.")
    return value


def resolve_bucket_name(config: Any, require: bool = True) -> str:
    """Resolve the GCS bucket from Kaggle Secrets."""
    secret_name = str(
        cfg_value(config, "GCS_BUCKET_SECRET_NAME", "GCS_BUCKET") or ""
    ).strip()
    strict = bool(cfg_value(config, "REQUIRE_KAGGLE_GCS_SECRETS", True))

    if strict:
        resolved = read_kaggle_secret(secret_name, required=require)
    else:
        resolved = (
            read_kaggle_secret(secret_name, required=False)
            or os.environ.get("GCS_BUCKET", "")
            or str(cfg_value(config, "GCS_BUCKET", "") or "")
        ).strip()

    if resolved.startswith("gs://"):
        resolved = resolved[len("gs://"):].split("/", 1)[0]
    resolved = resolved.strip().strip("/")

    if require and not resolved:
        raise RuntimeError(
            f"Missing GCS bucket. Add Kaggle Secret {secret_name!r}."
        )
    return resolved


def make_storage_client(config: Any):
    """Create a GCS client from a service-account"""
    from google.cloud import storage
    from google.oauth2 import service_account

    secret_name = str(
        cfg_value(
            config,
            "GCS_CREDENTIALS_JSON_SECRET_NAME",
            "GCS_CREDENTIALS_JSON",
        )
        or ""
    ).strip()
    strict = bool(cfg_value(config, "REQUIRE_KAGGLE_GCS_SECRETS", True))

    if strict:
        credentials_json = read_kaggle_secret(secret_name, required=True)
    else:
        credentials_json = (
            read_kaggle_secret(secret_name, required=False)
            or os.environ.get("GCS_CREDENTIALS_JSON", "")
        ).strip()

    if not credentials_json:
        if strict:
            raise RuntimeError(
                f"Missing service-account JSON in Kaggle Secret {secret_name!r}."
            )
        return storage.Client()

    try:
        credentials_info = json.loads(credentials_json)
    except json.JSONDecodeError as exc:
        raise RuntimeError(
            f"Kaggle Secret {secret_name!r} is not valid JSON."
        ) from exc

    required_fields = {"type", "project_id", "private_key", "client_email"}
    missing_fields = sorted(required_fields - set(credentials_info))
    if missing_fields:
        raise RuntimeError(
            f"Kaggle Secret {secret_name!r} is missing fields: {missing_fields}"
        )

    credentials = service_account.Credentials.from_service_account_info(
        credentials_info
    )
    return storage.Client(
        project=credentials_info.get("project_id"),
        credentials=credentials,
    )


### Format

In [ ]:
def parse_gcs_uri(uri: str) -> tuple[str, str]:
    """Parse gs://bucket/object into bucket and object name."""
    if not str(uri).startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got {uri}")
    bucket, _, blob = str(uri)[5:].partition("/")
    if not bucket or not blob:
        raise ValueError(f"Invalid GCS URI: {uri}")
    return bucket, blob


def sanitize_partition_value(value: Any) -> str:
    """Make a stable value safe for use in a Hive-style GCS partition."""
    text = str(value or "unknown").strip()
    text = re.sub(r"[^A-Za-z0-9._-]+", "-", text)
    return text.strip("-") or "unknown"


def output_base_prefix(
    config: Any,
    batch_id: str,
    model_key: str | None = None,
) -> str:
    """Build the model-level base prefix without data/ or runs/."""
    resolved_model_key = model_key or str(
        cfg_value(config, "ACTIVE_MODEL_KEY", "unknown")
    )
    return (
        f"{normalize_prefix(cfg_value(config, 'OUTPUT_PREFIX', 'features/extractors'))}/"
        f"dataset={sanitize_partition_value(cfg_value(config, 'DATASET_ID'))}/"
        f"batch={sanitize_partition_value(batch_id)}/"
        f"frame_profile={sanitize_partition_value(cfg_value(config, 'PROFILE_VERSION'))}/"
        f"extractor={sanitize_partition_value(cfg_value(config, 'EXTRACTOR_NAME'))}/"
        f"extractor_version={sanitize_partition_value(cfg_value(config, 'EXTRACTOR_VERSION'))}/"
        f"model={sanitize_partition_value(resolved_model_key)}/"
    )


def make_data_prefix(
    config: Any,
    batch_id: str,
    model_key: str | None = None,
) -> str:
    """Build the canonical data/ prefix."""
    return output_base_prefix(config, batch_id, model_key) + "data/"


def make_output_prefix(
    config: Any,
    batch_id: str,
    run_id: str,
    model_key: str | None = None,
) -> str:
    """Build the runs/run_id=.../ prefix."""
    return (
        output_base_prefix(config, batch_id, model_key)
        + f"runs/run_id={sanitize_partition_value(run_id)}/"
    )


In [ ]:
def make_run_layout(
    config: Any,
    batch_id: str,
    run_kind: str,
    model_key: str | None = None,
) -> RunLayout:
    """Create local directories and partitioned GCS paths for one run."""
    resolved_model_key = model_key or str(
        cfg_value(config, "ACTIVE_MODEL_KEY", "unknown")
    )
    run_id = make_run_id(run_kind)
    run_dir = (
        Path(str(cfg_value(config, "RUN_ROOT", "/kaggle/working/feature_extractor_runs")))
        / run_id
    )
    frames_dir = run_dir / "frames"
    artifacts_dir = run_dir / "artifacts"
    frames_dir.mkdir(parents=True, exist_ok=True)
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    model_base = output_base_prefix(
        config,
        batch_id,
        resolved_model_key,
    )
    data_prefix = model_base + "data/"
    runs_prefix = model_base + "runs/"
    run_prefix = runs_prefix + f"run_id={sanitize_partition_value(run_id)}/"
    latest_name = str(
        cfg_value(config, "LATEST_POINTER_FILENAME", "latest.json")
    )

    return RunLayout(
        run_id=run_id,
        batch_id=str(batch_id),
        model_key=resolved_model_key,
        run_dir=run_dir,
        frames_dir=frames_dir,
        artifacts_dir=artifacts_dir,
        model_base_prefix=model_base,
        data_prefix=data_prefix,
        runs_prefix=runs_prefix,
        output_prefix=run_prefix,
        latest_pointer_blob=model_base + latest_name,
        annotations_path=artifacts_dir / "annotations.jsonl",
        errors_path=artifacts_dir / "errors.jsonl",
        metrics_path=artifacts_dir / "metrics.csv",
        summary_path=artifacts_dir / "summary.json",
        config_path=artifacts_dir / "config.json",
        publish_manifest_path=artifacts_dir / "manifest.json",
        published_annotations_path=artifacts_dir / "published_annotations.jsonl",
        log_path=run_dir / "run.log",
    )


def setup_logging(layout: RunLayout, verbose: bool = False) -> logging.Logger:
    """Configure console and file logging for a notebook run."""
    logger = logging.getLogger(str(cfg_value(cfg, "EXTRACTOR_NAME", "feature_extractor")))
    logger.setLevel(logging.DEBUG if verbose else logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    stream = logging.StreamHandler()
    stream.setFormatter(fmt)
    file_handler = logging.FileHandler(layout.log_path, encoding="utf-8")
    file_handler.setFormatter(fmt)
    logger.addHandler(stream)
    logger.addHandler(file_handler)
    return logger


def upload_file(bucket: Any, local_path: Path, object_key: str, content_type: str = "application/octet-stream") -> None:
    """Upload one local file to GCS."""
    bucket.blob(object_key).upload_from_filename(str(local_path), content_type=content_type, timeout=900)


def upload_text(bucket: Any, text: str, object_key: str, content_type: str = "text/plain") -> None:
    """Upload text content to GCS."""
    bucket.blob(object_key).upload_from_string(text, content_type=content_type, timeout=300)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    """Write a JSON object to disk with UTF-8 encoding."""
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def append_jsonl(path: Path, records: Iterable[dict[str, Any]]) -> int:
    """Append JSONL records to a local file and return the row count."""
    count = 0
    with path.open("a", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
    return count


def append_metric(path: Path, row: dict[str, Any]) -> None:
    """Append one row to metrics.csv, creating the header if needed."""
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def latest_blob_name(bucket: Any, prefix: str, suffix: str) -> str | None:
    """Return the latest blob under prefix with the requested suffix."""
    blobs = [blob for blob in bucket.list_blobs(prefix=prefix) if blob.name.endswith(suffix)]
    if not blobs:
        return None
    blobs.sort(key=lambda b: (b.updated or datetime.min.replace(tzinfo=timezone.utc), b.name), reverse=True)
    return blobs[0].name


def find_manifest_blobs(config: Any, bucket: Any, batches: list[str]) -> list[str]:
    """Find GCS manifest files for selected batches."""
    explicit = str(cfg_value(config, "INPUT_MANIFEST_URI", "") or "").strip()
    if explicit:
        parsed_bucket, blob_name = parse_gcs_uri(explicit)
        if parsed_bucket != bucket.name:
            raise ValueError(f"INPUT_MANIFEST_URI bucket {parsed_bucket} does not match {bucket.name}")
        return [blob_name]
    blobs: list[str] = []
    base = normalize_prefix(cfg_value(config, "MANIFESTS_PREFIX", "processed/keyframes_manifests"))
    for batch_id in batches:
        prefix = f"{base}/dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/profile={cfg_value(config, 'PROFILE_VERSION')}/"
        name = latest_blob_name(bucket, prefix, "shot_segments.csv") or latest_blob_name(bucket, prefix, "frames_manifest.jsonl")
        if name is None:
            raise FileNotFoundError(f"No shot_segments.csv or frames_manifest.jsonl found under gs://{bucket.name}/{prefix}")
        blobs.append(name)
    return blobs


def read_manifest_blob(bucket: Any, blob_name: str) -> pd.DataFrame:
    """Read a CSV or JSONL frame manifest from GCS into a DataFrame."""
    text = bucket.blob(blob_name).download_as_text(timeout=900)
    if blob_name.endswith(".csv"):
        return pd.read_csv(StringIO(text))
    rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    return pd.DataFrame(rows)


def normalize_manifest_records(df: pd.DataFrame, config: Any, batch_id: str | None = None) -> list[dict[str, Any]]:
    """Normalize frame manifest columns into the extractor record contract."""
    if df.empty:
        return []
    df = df.copy()
    if "saved" in df.columns:
        df = df[df["saved"].astype(str).str.lower().isin(["true", "1", "yes"])]
    records: list[dict[str, Any]] = []
    for _, row in df.iterrows():
        image_gcs_uri = str(row.get("image_gcs_uri") or row.get("gcs_uri") or row.get("image_uri") or "").strip()
        if not image_gcs_uri:
            continue
        video_id = str(row.get("video_id") or Path(image_gcs_uri).parent.name).strip()
        frame_idx = int(float(row.get("frame_idx", 0) or 0))
        keyframe_id = str(row.get("keyframe_id") or f"{video_id}_F{frame_idx:06d}")
        records.append({
            "dataset_id": str(row.get("dataset_id") or cfg_value(config, "DATASET_ID")),
            "batch_id": str(row.get("batch_id") or batch_id or "").strip(),
            "video_id": video_id,
            "video_name": str(row.get("video_name") or ""),
            "shot_id": str(row.get("shot_id") or ""),
            "shot_start_frame": int(float(row.get("shot_start_frame", 0) or 0)),
            "shot_end_frame": int(float(row.get("shot_end_frame", 0) or 0)),
            "frame_type": str(row.get("frame_type") or ""),
            "frame_idx": frame_idx,
            "frame_sec": float(row.get("frame_sec", row.get("timestamp", 0)) or 0),
            "timestamp_ms": int(float(row.get("frame_sec", 0) or 0) * 1000),
            "keyframe_id": keyframe_id,
            "image_rel_path": str(row.get("image_rel_path") or f"{video_id}/{Path(image_gcs_uri).name}"),
            "image_gcs_uri": image_gcs_uri,
            "image_storage_key": str(row.get("image_storage_key") or parse_gcs_uri(image_gcs_uri)[1]),
            "fps": float(row.get("fps", 0) or 0),
            "profile_version": str(row.get("profile_version") or cfg_value(config, "PROFILE_VERSION")),
        })
    return records


def discover_frame_records(config: Any, bucket: Any, batches: list[str], max_frames: int | None = None) -> list[dict[str, Any]]:
    """Discover and normalize frame records for selected logical batches."""
    all_records: list[dict[str, Any]] = []
    for blob_name in find_manifest_blobs(config, bucket, batches):
        batch_hint = next((part.split("=", 1)[1] for part in blob_name.split("/") if part.startswith("batch=")), None)
        all_records.extend(normalize_manifest_records(read_manifest_blob(bucket, blob_name), config, batch_hint))
    all_records.sort(key=lambda r: (r["batch_id"], r["video_id"], r["frame_idx"], r["keyframe_id"]))
    return all_records[: int(max_frames)] if max_frames is not None else all_records


def local_frame_path(layout: RunLayout, record: dict[str, Any]) -> Path:
    """Return the local scratch path for one frame record."""
    return layout.frames_dir / record["video_id"] / Path(record["image_gcs_uri"]).name


def download_one_frame(record: dict[str, Any], layout: RunLayout, client: Any) -> dict[str, Any]:
    """Download one frame from GCS into local scratch and return an updated record."""
    started = time.perf_counter()
    bucket_name, blob_name = parse_gcs_uri(record["image_gcs_uri"])
    out_path = local_frame_path(layout, record)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if not out_path.exists() or out_path.stat().st_size == 0:
        client.bucket(bucket_name).blob(blob_name).download_to_filename(str(out_path), timeout=900)
    updated = dict(record)
    updated["local_image_path"] = str(out_path)
    updated["download_ms"] = int((time.perf_counter() - started) * 1000)
    return updated


def download_frames(records: list[dict[str, Any]], layout: RunLayout, client: Any, config: Any) -> list[dict[str, Any]]:
    """Download many frames concurrently from GCS."""
    downloaded: list[dict[str, Any]] = []
    workers = max(1, int(cfg_value(config, "DOWNLOAD_WORKERS", 8)))
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="gcs-download") as pool:
        futures = [pool.submit(download_one_frame, record, layout, client) for record in records]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading frames", disable=not cfg_value(config, "USE_TQDM", True)):
            downloaded.append(future.result())
    downloaded.sort(key=lambda r: (r["batch_id"], r["video_id"], r["frame_idx"], r["keyframe_id"]))
    return downloaded


def iter_batches(items: list[Any], batch_size: int) -> Iterable[list[Any]]:
    """Yield fixed-size batches from a list."""
    batch_size = max(1, int(batch_size))
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def base_annotation(record: dict[str, Any], config: Any, kind: str, run_id: str) -> dict[str, Any]:
    """Create the shared frame annotation JSON payload."""
    return {
        "dataset_id": record["dataset_id"], "batch_id": record["batch_id"], "video_id": record["video_id"],
        "keyframe_id": record["keyframe_id"], "frame_id": record["keyframe_id"], "shot_id": record.get("shot_id", ""),
        "frame_idx": record["frame_idx"], "frame_sec": record["frame_sec"], "timestamp_ms": record.get("timestamp_ms", int(float(record.get("frame_sec", 0)) * 1000)),
        "frame_type": record.get("frame_type", ""), "image_gcs_uri": record["image_gcs_uri"], "image_storage_key": record.get("image_storage_key", ""),
        "kind": kind, "caption": None, "ocr_texts": [], "detected_objects": [], "object_counts": {}, "detections": [],
        "text_value": None, "json_value": {}, "confidence": 1.0, "model_version": str(cfg_value(config, "MODEL_VERSION", "unknown")),
        "annotation_version": str(cfg_value(config, "ANNOTATION_VERSION", cfg_value(config, "EXTRACTOR_VERSION", "v1"))), "run_id": run_id, "created_at": utc_now(),
    }


def _jsonable(value: Any) -> Any:
    """Convert config values to JSON-safe objects without exposing secrets."""
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_jsonable(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return repr(value)


def build_config_snapshot(config: Any) -> dict[str, Any]:
    """Build a reproducible config snapshot; secret values are never included."""
    blocked = {
        "GCS_BUCKET",
        "GCS_CREDENTIALS_JSON",
        "GCS_CREDENTIALS_FILE",
    }
    snapshot: dict[str, Any] = {}
    for name, value in vars(config).items():
        if not str(name).isupper() or name in blocked:
            continue
        snapshot[str(name)] = _jsonable(value)
    snapshot["created_at"] = utc_now()
    snapshot["security_note"] = (
        "GCS secret values are read at runtime and are not stored in this file."
    )
    return snapshot


def _read_jsonl_rows(text: str) -> list[dict[str, Any]]:
    """Parse valid JSON objects from JSONL text."""
    rows: list[dict[str, Any]] = []
    for line in str(text or "").splitlines():
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except Exception:
            continue
        if isinstance(row, dict):
            rows.append(row)
    return rows


def _read_local_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    return _read_jsonl_rows(path.read_text(encoding="utf-8"))


def merge_annotations_for_publish(
    bucket: Any,
    layout: RunLayout,
    config: Any,
) -> tuple[Path, int]:
    """Merge canonical and current annotations, then deduplicate by keyframe_id."""
    canonical_name = str(
        cfg_value(config, "CANONICAL_ANNOTATIONS_FILENAME", "annotations.jsonl")
    )
    canonical_blob_name = layout.data_prefix + canonical_name
    overwrite = bool(cfg_value(config, "OVERWRITE", False))
    merge_existing = bool(
        cfg_value(config, "MERGE_WITH_EXISTING_CANONICAL", True)
    )

    existing_rows: list[dict[str, Any]] = []
    canonical_blob = bucket.blob(canonical_blob_name)
    if merge_existing and not overwrite and canonical_blob.exists():
        existing_rows = _read_jsonl_rows(
            canonical_blob.download_as_text(timeout=900)
        )

    current_rows = _read_local_jsonl(layout.annotations_path)

    keyed: dict[str, dict[str, Any]] = {}
    unkeyed: list[dict[str, Any]] = []
    for row in [*existing_rows, *current_rows]:
        key = str(row.get("keyframe_id") or row.get("frame_id") or "").strip()
        if key:
            keyed[key] = row  # Current run overwrites an older record with same key.
        else:
            unkeyed.append(row)

    merged_rows = [
        *sorted(
            keyed.values(),
            key=lambda row: (
                str(row.get("batch_id", "")),
                str(row.get("video_id", "")),
                int(row.get("frame_idx", 0) or 0),
                str(row.get("keyframe_id", "")),
            ),
        ),
        *unkeyed,
    ]

    with layout.published_annotations_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        for row in merged_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

    return layout.published_annotations_path, len(merged_rows)


def upload_run_artifacts(
    bucket: Any,
    layout: RunLayout,
    success: bool,
) -> None:
    """Upload lineage artifacts under runs/run_id=.../."""
    files = [
        (layout.annotations_path, "application/x-ndjson"),
        (layout.errors_path, "application/x-ndjson"),
        (layout.metrics_path, "text/csv"),
        (layout.summary_path, "application/json"),
        (layout.config_path, "application/json"),
        (layout.log_path, "text/plain"),
    ]
    for path, content_type in files:
        if path.exists():
            upload_file(
                bucket,
                path,
                layout.output_prefix + path.name,
                content_type,
            )
    if success:
        upload_text(
            bucket,
            "",
            layout.output_prefix + "_SUCCESS",
            "text/plain",
        )


def publish_canonical_data(
    bucket: Any,
    layout: RunLayout,
    config: Any,
    summary: dict[str, Any],
) -> dict[str, Any]:
    """Atomically publish canonical annotations, manifest, marker, and pointer."""
    annotations_path, record_count = merge_annotations_for_publish(
        bucket,
        layout,
        config,
    )
    annotations_name = str(
        cfg_value(config, "CANONICAL_ANNOTATIONS_FILENAME", "annotations.jsonl")
    )
    manifest_name = str(
        cfg_value(config, "CANONICAL_MANIFEST_FILENAME", "manifest.json")
    )

    success_blob = bucket.blob(layout.data_prefix + "_SUCCESS")
    if success_blob.exists():
        success_blob.delete()

    canonical_annotations_blob = layout.data_prefix + annotations_name
    canonical_manifest_blob = layout.data_prefix + manifest_name

    manifest = {
        "status": "SUCCESS",
        "dataset_id": cfg_value(config, "DATASET_ID"),
        "batch_id": layout.batch_id,
        "frame_profile": cfg_value(config, "PROFILE_VERSION"),
        "extractor": cfg_value(config, "EXTRACTOR_NAME"),
        "extractor_version": cfg_value(config, "EXTRACTOR_VERSION"),
        "model_key": layout.model_key,
        "model_id": summary.get("model_id"),
        "active_run_id": layout.run_id,
        "record_count": record_count,
        "annotations_uri": f"gs://{bucket.name}/{canonical_annotations_blob}",
        "run_uri": f"gs://{bucket.name}/{layout.output_prefix}",
        "published_at": utc_now(),
    }
    write_json(layout.publish_manifest_path, manifest)

    upload_file(
        bucket,
        annotations_path,
        canonical_annotations_blob,
        "application/x-ndjson",
    )
    upload_file(
        bucket,
        layout.publish_manifest_path,
        canonical_manifest_blob,
        "application/json",
    )
    upload_text(
        bucket,
        "",
        layout.data_prefix + "_SUCCESS",
        "text/plain",
    )

    if bool(cfg_value(config, "WRITE_LATEST_POINTER", True)):
        latest_pointer = {
            **manifest,
            "data_prefix": f"gs://{bucket.name}/{layout.data_prefix}",
            "latest_pointer_uri": (
                f"gs://{bucket.name}/{layout.latest_pointer_blob}"
            ),
        }
        upload_text(
            bucket,
            json.dumps(latest_pointer, ensure_ascii=False, indent=2),
            layout.latest_pointer_blob,
            "application/json",
        )

    return manifest


def find_resume_annotations_blob(
    config: Any,
    bucket: Any,
    batches: list[str],
) -> str | None:
    """Use the canonical data/annotations.jsonl as the default resume source."""
    explicit = str(
        cfg_value(config, "RESUME_ANNOTATIONS_URI", "") or ""
    ).strip()
    if explicit:
        parsed_bucket, blob_name = parse_gcs_uri(explicit)
        if parsed_bucket != bucket.name:
            raise ValueError(
                f"RESUME_ANNOTATIONS_URI bucket {parsed_bucket} "
                f"does not match {bucket.name}"
            )
        return blob_name

    batch_partition = batches[0] if len(batches) == 1 else "all"
    canonical_name = str(
        cfg_value(config, "CANONICAL_ANNOTATIONS_FILENAME", "annotations.jsonl")
    )
    blob_name = (
        make_data_prefix(
            config,
            batch_partition,
            str(cfg_value(config, "ACTIVE_MODEL_KEY", "unknown")),
        )
        + canonical_name
    )
    return blob_name if bucket.blob(blob_name).exists() else None


def load_processed_keyframes(
    config: Any,
    bucket: Any,
    batches: list[str],
) -> set[str]:
    """Load successful keyframe IDs from canonical annotations."""
    if (
        not cfg_value(config, "SKIP_EXISTING", True)
        or cfg_value(config, "OVERWRITE", False)
    ):
        return set()

    blob_name = find_resume_annotations_blob(config, bucket, batches)
    if not blob_name:
        return set()

    text = bucket.blob(blob_name).download_as_text(timeout=900)
    processed: set[str] = set()
    for row in _read_jsonl_rows(text):
        if not row.get("error") and row.get("keyframe_id"):
            processed.add(str(row["keyframe_id"]))
    return processed


def dry_run(config: Any, max_frames: int | None = None) -> dict[str, Any]:
    """Validate Kaggle Secrets, GCS access, manifests, and planned paths."""
    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    batches = [
        str(batch).upper()
        for batch in cfg_value(config, "BATCHES", [])
    ]
    records = discover_frame_records(
        config,
        bucket,
        batches,
        max_frames=max_frames,
    )
    model_key = str(cfg_value(config, "ACTIVE_MODEL_KEY", "unknown"))
    planned_layout = {}
    for batch_id in batches:
        model_base = output_base_prefix(config, batch_id, model_key)
        planned_layout[batch_id] = {
            "model_base": f"gs://{bucket.name}/{model_base}",
            "canonical_data": f"gs://{bucket.name}/{model_base}data/",
            "runs": f"gs://{bucket.name}/{model_base}runs/",
            "latest_pointer": f"gs://{bucket.name}/{model_base}latest.json",
        }

    return {
        "status": "DRY_RUN_OK",
        "bucket": bucket.name,
        "batches": batches,
        "active_model_key": model_key,
        "planned_frames": len(records),
        "planned_gcs_layout": planned_layout,
        "sample_records": records[:5],
    }


## 3b. Captioning Model Registry And Extraction Logic

**Note:** Full/Demo chỉ load `ACTIVE_MODEL_KEY`. Benchmark load từng model tuần tự, đo thời gian và VRAM, sau đó giải phóng model trước khi chuyển sang model tiếp theo.


In [ ]:
import gc
from pathlib import Path
from typing import Any


def configure_transformers_progress(config: Any) -> None:
    """Enable or disable Hugging Face download progress bars."""
    show_progress = bool(cfg_value(config, "SHOW_HF_DOWNLOAD_PROGRESS", True))
    try:
        from huggingface_hub.utils import disable_progress_bars, enable_progress_bars
        from transformers.utils import logging as transformers_logging
        if show_progress:
            enable_progress_bars()
            transformers_logging.enable_progress_bar()
        else:
            disable_progress_bars()
            transformers_logging.disable_progress_bar()
    except Exception:
        # Progress configuration should never prevent model loading.
        pass

def get_model_spec(config: Any, model_key: str) -> dict[str, Any]:
    """Return a defensive copy of one registry entry."""
    registry = cfg_value(config, "MODEL_REGISTRY", {})
    if model_key not in registry:
        raise KeyError(f"Model key {model_key!r} is not in MODEL_REGISTRY.")
    spec = dict(registry[model_key])
    spec["model_key"] = model_key
    return spec


def resolve_torch_device(config: Any) -> str:
    """Resolve the requested model device."""
    import torch

    requested = str(cfg_value(config, "DEVICE", "auto")).lower()
    if requested == "cpu":
        return "cpu"
    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("DEVICE='cuda' but CUDA is unavailable.")
        return "cuda"
    return "cuda" if torch.cuda.is_available() else "cpu"


def resolve_torch_dtype(device: str):
    """Use BF16 when supported, otherwise FP16 on CUDA and FP32 on CPU."""
    import torch

    if device == "cpu":
        return torch.float32
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def _quantization_config(spec: dict[str, Any], device: str):
    """Build a BitsAndBytesConfig or return None."""
    import torch
    from transformers import BitsAndBytesConfig

    mode = str(spec.get("quantization", "none")).lower()
    if mode == "none":
        return None
    if device != "cuda":
        raise RuntimeError(f"quantization={mode!r} requires CUDA.")
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    if mode == "4bit":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype,
        )
    if mode == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    raise ValueError(f"Unsupported quantization mode: {mode}")


def _first_model_device(model: Any):
    """Find the device used for input tensors."""
    try:
        return model.device
    except Exception:
        return next(model.parameters()).device


def synchronize_cuda(gpu_ids: list[int] | None = None) -> None:
    """Synchronize selected CUDA devices once."""
    import torch

    if not torch.cuda.is_available():
        return
    ids = gpu_ids if gpu_ids is not None else list(range(torch.cuda.device_count()))
    for gpu_id in ids:
        torch.cuda.synchronize(int(gpu_id))


def resolve_gpu_ids(config: Any) -> list[int]:
    """Return valid visible GPU IDs requested by the configuration."""
    import torch

    if not torch.cuda.is_available():
        return []

    visible_count = torch.cuda.device_count()
    requested = cfg_value(config, "GPU_IDS", list(range(visible_count)))
    if requested is None:
        requested = list(range(visible_count))

    gpu_ids: list[int] = []
    for raw_id in requested:
        gpu_id = int(raw_id)
        if gpu_id < 0 or gpu_id >= visible_count:
            raise ValueError(
                f"GPU_IDS contains {gpu_id}, but visible GPU IDs are "
                f"0..{visible_count - 1}."
            )
        if gpu_id not in gpu_ids:
            gpu_ids.append(gpu_id)

    if not gpu_ids:
        gpu_ids = [0]
    return gpu_ids


def resolve_gpu_execution_mode(config: Any) -> str:
    """Validate the requested GPU execution strategy."""
    mode = str(
        cfg_value(config, "GPU_EXECUTION_MODE", "single")
    ).strip().lower()
    allowed = {"replicated", "single", "sharded"}
    if mode not in allowed:
        raise ValueError(
            f"GPU_EXECUTION_MODE={mode!r}; expected one of {sorted(allowed)}."
        )
    return mode


def reset_cuda_peak_memory(gpu_ids: list[int] | None = None) -> None:
    """Reset CUDA peak-memory counters on selected visible GPUs."""
    import torch

    if not torch.cuda.is_available():
        return

    ids = gpu_ids if gpu_ids is not None else list(range(torch.cuda.device_count()))
    synchronize_cuda(ids)
    for gpu_id in ids:
        torch.cuda.reset_peak_memory_stats(int(gpu_id))


def cuda_memory_metrics(
    gpu_ids: list[int] | None = None,
    synchronize: bool = True,
) -> dict[str, float]:
    """Return aggregate and per-GPU CUDA memory metrics in GiB."""
    import torch

    if not torch.cuda.is_available():
        return {
            "vram_allocated_gb": 0.0,
            "vram_reserved_gb": 0.0,
            "peak_vram_allocated_gb": 0.0,
            "peak_vram_reserved_gb": 0.0,
        }

    ids = gpu_ids if gpu_ids is not None else list(range(torch.cuda.device_count()))
    ids = [int(gpu_id) for gpu_id in ids]
    if synchronize:
        synchronize_cuda(ids)

    scale = 1024 ** 3
    result: dict[str, float] = {
        "vram_allocated_gb": round(
            sum(torch.cuda.memory_allocated(i) for i in ids) / scale,
            4,
        ),
        "vram_reserved_gb": round(
            sum(torch.cuda.memory_reserved(i) for i in ids) / scale,
            4,
        ),
        "peak_vram_allocated_gb": round(
            sum(torch.cuda.max_memory_allocated(i) for i in ids) / scale,
            4,
        ),
        "peak_vram_reserved_gb": round(
            sum(torch.cuda.max_memory_reserved(i) for i in ids) / scale,
            4,
        ),
    }

    for gpu_id in ids:
        result[f"gpu_{gpu_id}_allocated_gb"] = round(
            torch.cuda.memory_allocated(gpu_id) / scale,
            4,
        )
        result[f"gpu_{gpu_id}_reserved_gb"] = round(
            torch.cuda.memory_reserved(gpu_id) / scale,
            4,
        )
        result[f"gpu_{gpu_id}_peak_allocated_gb"] = round(
            torch.cuda.max_memory_allocated(gpu_id) / scale,
            4,
        )
        result[f"gpu_{gpu_id}_peak_reserved_gb"] = round(
            torch.cuda.max_memory_reserved(gpu_id) / scale,
            4,
        )
    return result




# Model
def load_caption_model(
    config: Any,
    logger: logging.Logger,
    model_key: str | None = None,
    gpu_id: int | None = None,
    force_sharded: bool = False,
) -> dict[str, Any]:
    """Load one caption model on one GPU, CPU, or an auto-sharded map."""
    import torch
    from transformers import AutoProcessor

    model_key = model_key or str(cfg_value(config, "ACTIVE_MODEL_KEY"))
    spec = get_model_spec(config, model_key)
    backend = str(spec["backend"])
    model_id = str(spec["hf_id"])
    device = resolve_torch_device(config)
    dtype = resolve_torch_dtype(device)
    quantization_config = _quantization_config(spec, device)

    configure_transformers_progress(config)

    common_kwargs: dict[str, Any] = {
        "cache_dir": str(cfg_value(config, "HF_CACHE_DIR", "")) or None,
        "low_cpu_mem_usage": bool(cfg_value(config, "LOW_CPU_MEM_USAGE", True)),
        "trust_remote_code": bool(cfg_value(config, "TRUST_REMOTE_CODE", False)),
        "local_files_only": bool(
            cfg_value(config, "TRANSFORMERS_LOCAL_FILES_ONLY", False)
        ),
    }
    common_kwargs = {
        key: value
        for key, value in common_kwargs.items()
        if value is not None
    }

    assigned_gpu_id: int | None = None
    if device == "cuda":
        if force_sharded:
            common_kwargs["device_map"] = "auto"
        else:
            if gpu_id is None:
                gpu_id = resolve_gpu_ids(config)[0]
            assigned_gpu_id = int(gpu_id)
            common_kwargs["device_map"] = {"": assigned_gpu_id}
        common_kwargs["torch_dtype"] = dtype
    else:
        common_kwargs["torch_dtype"] = torch.float32

    if quantization_config is not None:
        common_kwargs["quantization_config"] = quantization_config

    if bool(cfg_value(config, "USE_FLASH_ATTENTION_2", False)) and device == "cuda":
        common_kwargs["attn_implementation"] = "flash_attention_2"

    placement = (
        "auto-sharded"
        if force_sharded and device == "cuda"
        else f"cuda:{assigned_gpu_id}"
        if assigned_gpu_id is not None
        else device
    )
    logger.info(
        "Loading model_key=%s backend=%s model_id=%s placement=%s "
        "dtype=%s quantization=%s",
        model_key,
        backend,
        model_id,
        placement,
        dtype,
        spec.get("quantization", "none"),
    )

    if assigned_gpu_id is not None:
        torch.cuda.set_device(assigned_gpu_id)

    if backend == "qwen25_vl":
        from transformers import Qwen2_5_VLForConditionalGeneration

        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id,
            **common_kwargs,
        )
    elif backend == "qwen3_vl":
        from transformers import Qwen3VLForConditionalGeneration

        model = Qwen3VLForConditionalGeneration.from_pretrained(
            model_id,
            **common_kwargs,
        )
    elif backend == "blip2":
        from transformers import Blip2ForConditionalGeneration

        model = Blip2ForConditionalGeneration.from_pretrained(
            model_id,
            **common_kwargs,
        )
        if device == "cpu":
            model = model.to("cpu")
    else:
        raise ValueError(
            f"Unsupported backend={backend!r}. "
            "Add a loader branch in load_caption_model() and an inference branch "
            "in generate_caption_batch()."
        )

    model.eval()
    processor = AutoProcessor.from_pretrained(
        model_id,
        cache_dir=str(cfg_value(config, "HF_CACHE_DIR", "")) or None,
        trust_remote_code=bool(cfg_value(config, "TRUST_REMOTE_CODE", False)),
        local_files_only=bool(
            cfg_value(config, "TRANSFORMERS_LOCAL_FILES_ONLY", False)
        ),
    )
    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        processor.tokenizer.padding_side = "left"

    config.MODEL_VERSION = model_id
    return {
        "model_key": model_key,
        "spec": spec,
        "backend": backend,
        "model_id": model_id,
        "model": model,
        "processor": processor,
        "device": device,
        "dtype": dtype,
        "gpu_id": assigned_gpu_id,
        "is_sharded": bool(force_sharded and device == "cuda"),
        "input_device": _first_model_device(model),
    }

def load_caption_model_pool(
    config: Any,
    logger: logging.Logger,
    model_key: str | None = None,
) -> dict[str, Any]:
    """Load one or more model contexts according to GPU_EXECUTION_MODE."""
    model_key = model_key or str(cfg_value(config, "ACTIVE_MODEL_KEY"))
    device = resolve_torch_device(config)
    mode = resolve_gpu_execution_mode(config)
    contexts: list[dict[str, Any]] = []
    executor = None

    try:
        if device != "cuda":
            mode = "single"
            contexts = [load_caption_model(config, logger, model_key)]
        elif mode == "replicated":
            gpu_ids = resolve_gpu_ids(config)
            contexts = [
                load_caption_model(
                    config,
                    logger,
                    model_key,
                    gpu_id=gpu_id,
                )
                for gpu_id in gpu_ids
            ]
            if len(contexts) > 1:
                executor = ThreadPoolExecutor(
                    max_workers=len(contexts),
                    thread_name_prefix="caption-gpu",
                )
        elif mode == "single":
            gpu_id = resolve_gpu_ids(config)[0]
            contexts = [
                load_caption_model(
                    config,
                    logger,
                    model_key,
                    gpu_id=gpu_id,
                )
            ]
        else:
            contexts = [
                load_caption_model(
                    config,
                    logger,
                    model_key,
                    force_sharded=True,
                )
            ]
    except Exception:
        for ctx in contexts:
            unload_caption_model(ctx)
        if executor is not None:
            executor.shutdown(wait=True, cancel_futures=True)
        raise

    per_context_batch_sizes = [
        max(1, int(ctx["spec"].get("batch_size", 1)))
        for ctx in contexts
    ]
    global_batch_size = (
        sum(per_context_batch_sizes)
        if mode == "replicated"
        else per_context_batch_sizes[0]
    )
    gpu_ids = [
        int(ctx["gpu_id"])
        for ctx in contexts
        if ctx.get("gpu_id") is not None
    ]
    if not gpu_ids and device == "cuda":
        gpu_ids = resolve_gpu_ids(config)

    logger.info(
        "Model pool ready: mode=%s replicas=%d gpu_ids=%s "
        "per_gpu_batches=%s global_batch_size=%d",
        mode,
        len(contexts),
        gpu_ids,
        per_context_batch_sizes,
        global_batch_size,
    )
    return {
        "model_key": model_key,
        "mode": mode,
        "contexts": contexts,
        "executor": executor,
        "gpu_ids": gpu_ids,
        "per_context_batch_sizes": per_context_batch_sizes,
        "global_batch_size": global_batch_size,
    }


def unload_caption_model_pool(pool: dict[str, Any] | None) -> None:
    """Shutdown inference workers and release every model replica."""
    if not pool:
        return

    executor = pool.get("executor")
    if executor is not None:
        executor.shutdown(wait=True, cancel_futures=True)

    for ctx in pool.get("contexts", []):
        unload_caption_model(ctx)
    pool.clear()



def unload_caption_model(ctx: dict[str, Any] | None) -> None:
    """Delete model objects and release cached CUDA memory."""
    import torch

    if ctx:
        ctx.pop("model", None)
        ctx.pop("processor", None)
        ctx.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        gc.collect()


def _chunked(items: list[Any], batch_size: int):
    """Yield fixed-size model batches."""
    for start in range(0, len(items), max(1, int(batch_size))):
        yield items[start:start + max(1, int(batch_size))]


def _image_uri(image_path: str | Path) -> str:
    """Convert a local image path to the file URI expected by qwen-vl-utils."""
    return Path(image_path).expanduser().resolve().as_uri()


def _build_qwen_messages(
    image_path: str | Path,
    spec: dict[str, Any],
) -> list[dict[str, Any]]:
    """Build one Qwen chat conversation."""
    image_content: dict[str, Any] = {
        "type": "image",
        "image": _image_uri(image_path),
    }
    max_pixels = spec.get("max_pixels")
    if max_pixels:
        image_content["max_pixels"] = int(max_pixels)
    return [
        {
            "role": "user",
            "content": [
                image_content,
                {"type": "text", "text": str(spec["prompt"])},
            ],
        }
    ]


def _move_batch_to_device(
    inputs: Any,
    device: Any,
    dtype: Any | None = None,
):
    """Move a BatchFeature/dict while preserving integer tensor dtypes."""
    import torch

    moved = {}
    for key, value in inputs.items():
        if torch.is_tensor(value):
            kwargs = {
                "device": device,
                "non_blocking": True,
            }
            if dtype is not None and torch.is_floating_point(value):
                kwargs["dtype"] = dtype
            moved[key] = value.to(**kwargs)
        else:
            moved[key] = value
    return moved


def _generate_qwen_batch(
    image_paths: list[str],
    ctx: dict[str, Any],
) -> list[str]:
    """Generate one Qwen batch with bounded parallel CPU preprocessing."""
    import torch
    from qwen_vl_utils import process_vision_info

    model = ctx["model"]
    processor = ctx["processor"]
    spec = ctx["spec"]
    backend = ctx["backend"]
    gpu_id = ctx.get("gpu_id")

    if gpu_id is not None:
        torch.cuda.set_device(int(gpu_id))

    conversations = [
        _build_qwen_messages(path, spec)
        for path in image_paths
    ]
    texts = [
        processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        for messages in conversations
    ]

    def process_one(messages: list[dict[str, Any]]):
        if backend == "qwen3_vl":
            patch_size = getattr(
                getattr(processor, "image_processor", None),
                "patch_size",
                16,
            )
            return process_vision_info(
                messages,
                image_patch_size=patch_size,
            )
        return process_vision_info(messages)

    worker_count = min(
        len(conversations),
        max(1, int(spec.get("preprocess_workers", 1))),
    )
    if worker_count > 1:
        with ThreadPoolExecutor(
            max_workers=worker_count,
            thread_name_prefix=(
                f"preprocess-gpu-{gpu_id}"
                if gpu_id is not None
                else "preprocess"
            ),
        ) as pool:
            processed = list(pool.map(process_one, conversations))
    else:
        processed = [process_one(messages) for messages in conversations]

    image_inputs: list[Any] = []
    video_inputs: list[Any] = []
    for sample_images, sample_videos in processed:
        if sample_images:
            image_inputs.extend(sample_images)
        if sample_videos:
            video_inputs.extend(sample_videos)

    processor_kwargs: dict[str, Any] = {
        "text": texts,
        "images": image_inputs or None,
        "videos": video_inputs or None,
        "return_tensors": "pt",
        "padding": True,
    }
    if backend == "qwen3_vl":
        # qwen-vl-utils already applied the requested max_pixels resize.
        processor_kwargs["do_resize"] = False

    inputs = processor(**processor_kwargs)
    inputs = _move_batch_to_device(
        inputs,
        ctx["input_device"],
    )

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=int(spec.get("max_new_tokens", 64)),
            do_sample=False,
            use_cache=True,
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            inputs["input_ids"],
            generated_ids,
        )
    ]
    return [
        text.strip()
        for text in processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
    ]


def _generate_blip2_batch(
    image_paths: list[str],
    ctx: dict[str, Any],
) -> list[str]:
    """Generate captions with BLIP-2 OPT."""
    import torch
    from PIL import Image

    model = ctx["model"]
    processor = ctx["processor"]
    spec = ctx["spec"]
    gpu_id = ctx.get("gpu_id")

    if gpu_id is not None:
        torch.cuda.set_device(int(gpu_id))

    images = []
    try:
        images = [Image.open(path).convert("RGB") for path in image_paths]
        inputs = processor(
            images=images,
            text=[str(spec["prompt"])] * len(images),
            return_tensors="pt",
            padding=True,
        )
        target_dtype = (
            ctx["dtype"]
            if ctx["device"] == "cuda"
            else torch.float32
        )
        inputs = _move_batch_to_device(
            inputs,
            ctx["input_device"],
            dtype=target_dtype,
        )

        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=int(spec.get("max_new_tokens", 80)),
                min_new_tokens=int(spec.get("min_new_tokens", 0)),
                num_beams=int(spec.get("num_beams", 1)),
                repetition_penalty=float(
                    spec.get("repetition_penalty", 1.0)
                ),
                length_penalty=float(spec.get("length_penalty", 1.0)),
                use_cache=True,
            )

        return [
            text.strip()
            for text in processor.batch_decode(
                generated_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
        ]
    finally:
        for image in images:
            image.close()


def generate_caption_batch(
    image_paths: list[str | Path],
    ctx: dict[str, Any],
) -> list[str]:
    """Caption a list of images using the backend registered in ctx."""
    image_paths = [str(path) for path in image_paths]
    if not image_paths:
        return []

    backend = ctx["backend"]
    batch_size = int(ctx["spec"].get("batch_size", 1))
    captions: list[str] = []

    for path_batch in _chunked(image_paths, batch_size):
        if backend in {"qwen25_vl", "qwen3_vl"}:
            captions.extend(_generate_qwen_batch(path_batch, ctx))
        elif backend == "blip2":
            captions.extend(_generate_blip2_batch(path_batch, ctx))
        else:
            raise ValueError(f"No inference implementation for backend={backend!r}")

    if len(captions) != len(image_paths):
        raise RuntimeError(
            f"Caption count mismatch: got {len(captions)} outputs "
            f"for {len(image_paths)} images."
        )
    return captions

def generate_caption_batch_parallel(
    image_paths: list[str | Path],
    model_pool: dict[str, Any],
) -> list[str]:
    """Run independent model replicas concurrently while preserving order."""
    image_paths = [str(path) for path in image_paths]
    if not image_paths:
        return []

    contexts = list(model_pool["contexts"])
    if len(contexts) == 1:
        return generate_caption_batch(image_paths, contexts[0])

    executor = model_pool.get("executor")
    if executor is None:
        raise RuntimeError(
            "Replicated model pool has multiple contexts but no executor."
        )

    capacities = [
        max(1, int(ctx["spec"].get("batch_size", 1)))
        for ctx in contexts
    ]
    global_capacity = sum(capacities)
    captions: list[str] = []

    # A round never exceeds the sum of all per-GPU capacities.
    for round_paths in _chunked(image_paths, global_capacity):
        # Balance even a partially filled round across GPUs. Each assignment
        # keeps original indexes so results can be restored to input order.
        assignments: list[list[tuple[int, str]]] = [
            [] for _ in contexts
        ]
        remaining_capacity = capacities.copy()
        next_context = 0

        for item_index, path in enumerate(round_paths):
            for _ in range(len(contexts)):
                candidate = next_context % len(contexts)
                next_context += 1
                if remaining_capacity[candidate] > 0:
                    assignments[candidate].append((item_index, path))
                    remaining_capacity[candidate] -= 1
                    break
            else:
                raise RuntimeError(
                    "No model replica had capacity for the current round."
                )

        futures = []
        for ctx, assignment in zip(contexts, assignments):
            if not assignment:
                continue
            indexes = [index for index, _ in assignment]
            shard_paths = [path for _, path in assignment]
            futures.append(
                (
                    indexes,
                    executor.submit(
                        generate_caption_batch,
                        shard_paths,
                        ctx,
                    ),
                )
            )

        round_outputs: list[tuple[int, str]] = []
        for indexes, future in futures:
            shard_captions = future.result()
            if len(shard_captions) != len(indexes):
                raise RuntimeError(
                    "A GPU replica returned a different number of captions "
                    "than assigned images."
                )
            round_outputs.extend(zip(indexes, shard_captions))

        captions.extend(
            caption
            for _, caption in sorted(
                round_outputs,
                key=lambda item: item[0],
            )
        )

    if len(captions) != len(image_paths):
        raise RuntimeError(
            f"Parallel caption count mismatch: got {len(captions)} outputs "
            f"for {len(image_paths)} images."
        )
    return captions



def extract_task_batch(
    records: list[dict[str, Any]],
    model_pool: dict[str, Any],
    config: Any,
    layout: RunLayout,
) -> list[dict[str, Any]]:
    """Create caption annotations for one downloaded frame batch."""
    captions = generate_caption_batch_parallel(
        [record["local_image_path"] for record in records],
        model_pool,
    )

    primary_ctx = model_pool["contexts"][0]
    outputs: list[dict[str, Any]] = []
    for record, caption in zip(records, captions):
        item = base_annotation(
            record,
            config,
            kind="captioning",
            run_id=layout.run_id,
        )
        item.update({
            "caption": caption,
            "text_value": caption or None,
            "json_value": {
                "caption": caption,
                "model_key": primary_ctx["model_key"],
                "model_id": primary_ctx["model_id"],
                "backend": primary_ctx["backend"],
                "quantization": primary_ctx["spec"].get(
                    "quantization",
                    "none",
                ),
                "gpu_execution_mode": model_pool["mode"],
            },
        })
        outputs.append(item)
    return outputs


def _select_benchmark_frames(
    records: list[dict[str, Any]],
    frame_limit: int = 5,
) -> list[dict[str, Any]]:
    """Select exactly frame_limit records spread across the manifest."""
    frame_limit = max(1, int(frame_limit))
    if len(records) < frame_limit:
        raise RuntimeError(
            f"Only found {len(records)} frames, but BENCHMARK_FRAME_LIMIT="
            f"{frame_limit}."
        )

    # Evenly spread the sample instead of taking five adjacent frames.
    indexes = np.linspace(
        0,
        len(records) - 1,
        num=frame_limit,
        dtype=int,
    )
    # np.linspace can repeat indexes for tiny inputs; preserve order and fill gaps.
    selected_indexes: list[int] = []
    for index in indexes.tolist():
        if index not in selected_indexes:
            selected_indexes.append(index)
    if len(selected_indexes) < frame_limit:
        for index in range(len(records)):
            if index not in selected_indexes:
                selected_indexes.append(index)
            if len(selected_indexes) == frame_limit:
                break

    return [records[index] for index in selected_indexes[:frame_limit]]


def _select_benchmark_records(
    records: list[dict[str, Any]],
    video_limit: int,
    frames_per_video: int,
) -> list[dict[str, Any]]:
    """Select representative records from distinct videos."""
    by_video: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_video.setdefault(str(record["video_id"]), []).append(record)

    selected: list[dict[str, Any]] = []
    for video_id in sorted(by_video)[: max(1, int(video_limit))]:
        candidates = sorted(
            by_video[video_id],
            key=lambda row: (row["frame_idx"], row["keyframe_id"]),
        )
        count = min(max(1, int(frames_per_video)), len(candidates))
        if count == 1:
            chosen = [candidates[len(candidates) // 2]]
        else:
            # Evenly spread representative frames across the video's keyframes.
            indexes = np.linspace(0, len(candidates) - 1, num=count, dtype=int)
            chosen = [candidates[index] for index in indexes]
        selected.extend(chosen)
    return selected


def benchmark_caption_models(
    config: Any,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Benchmark real batched captioning on the configured GPU strategy."""
    import torch

    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    batches = [
        str(batch).upper()
        for batch in cfg_value(config, "BENCHMARK_BATCHES", ["L21"])
    ]
    frame_limit = int(cfg_value(config, "BENCHMARK_FRAME_LIMIT", 32))
    model_keys = [
        str(model_key)
        for model_key in cfg_value(
            config,
            "BENCHMARK_MODEL_KEYS",
            [cfg_value(config, "ACTIVE_MODEL_KEY")],
        )
    ]

    all_records = discover_frame_records(
        config,
        bucket,
        batches,
        max_frames=None,
    )
    selected = _select_benchmark_frames(all_records, frame_limit)

    layout = make_run_layout(
        config,
        "benchmark",
        "benchmark",
        model_key="multi_model_benchmark",
    )
    logger = setup_logging(layout)

    selected = download_frames(selected, layout, client, config)
    logger.info(
        "Benchmark selected %d frames from batches=%s; models=%s",
        len(selected),
        batches,
        model_keys,
    )

    rows: list[dict[str, Any]] = []
    summary_rows: list[dict[str, Any]] = []

    model_bar = tqdm(
        model_keys,
        desc="Benchmark models",
        unit="model",
        disable=not bool(cfg_value(config, "USE_TQDM", True)),
    )

    for model_key in model_bar:
        model_bar.set_postfix_str(model_key)
        model_pool = None
        load_started = time.perf_counter()
        inference_seconds_total = 0.0

        try:
            requested_gpu_ids = (
                resolve_gpu_ids(config)
                if torch.cuda.is_available()
                else []
            )
            reset_cuda_peak_memory(requested_gpu_ids or None)

            tqdm.write(
                f"\nLoading {model_key}: "
                f"{get_model_spec(config, model_key)['hf_id']}"
            )
            model_pool = load_caption_model_pool(
                config,
                logger,
                model_key,
            )
            gpu_ids = model_pool["gpu_ids"]
            actual_execution_mode = model_pool["mode"]
            replica_count = len(model_pool["contexts"])
            global_batch_capacity = int(model_pool["global_batch_size"])
            if torch.cuda.is_available():
                synchronize_cuda(gpu_ids or None)
            load_seconds = time.perf_counter() - load_started
            loaded_memory = cuda_memory_metrics(
                gpu_ids or None,
                synchronize=False,
            )

            global_capacity = int(model_pool["global_batch_size"])
            requested_batch_size = int(
                cfg_value(config, "PIPELINE_BATCH_SIZE", global_capacity)
            )
            benchmark_batch_size = max(
                1,
                min(requested_batch_size, global_capacity),
            )

            if bool(cfg_value(config, "BENCHMARK_WARMUP", True)):
                warmup_records = selected[:benchmark_batch_size]
                generate_caption_batch_parallel(
                    [
                        record["local_image_path"]
                        for record in warmup_records
                    ],
                    model_pool,
                )
                if torch.cuda.is_available():
                    synchronize_cuda(gpu_ids or None)
                reset_cuda_peak_memory(gpu_ids or None)
                tqdm.write(
                    f"[{model_key}] warmup complete: "
                    f"{len(warmup_records)} frames"
                )

            frame_rows: list[dict[str, Any]] = []
            frame_bar = tqdm(
                total=len(selected),
                desc=f"{model_key}: batched captioning",
                unit="frame",
                leave=True,
                disable=not bool(cfg_value(config, "USE_TQDM", True)),
            )

            for batch_index, record_batch in enumerate(
                iter_batches(selected, benchmark_batch_size),
                start=1,
            ):
                image_paths = [
                    record["local_image_path"]
                    for record in record_batch
                ]

                if torch.cuda.is_available():
                    synchronize_cuda(gpu_ids or None)
                batch_started = time.perf_counter()

                captions = generate_caption_batch_parallel(
                    image_paths,
                    model_pool,
                )

                if torch.cuda.is_available():
                    synchronize_cuda(gpu_ids or None)
                batch_seconds = time.perf_counter() - batch_started
                inference_seconds_total += batch_seconds
                memory = cuda_memory_metrics(
                    gpu_ids or None,
                    synchronize=False,
                )
                average_frame_seconds = (
                    batch_seconds / len(record_batch)
                )

                for record, caption in zip(record_batch, captions):
                    row = {
                        "batch_id": record["batch_id"],
                        "video_id": record["video_id"],
                        "keyframe_id": record["keyframe_id"],
                        "frame_idx": record["frame_idx"],
                        "frame_sec": record["frame_sec"],
                        "image_gcs_uri": record["image_gcs_uri"],
                        "local_image_path": record["local_image_path"],
                        "model_key": model_pool["model_key"],
                        "model_id": model_pool["contexts"][0]["model_id"],
                        "backend": model_pool["contexts"][0]["backend"],
                        "quantization": model_pool["contexts"][0]["spec"].get(
                            "quantization",
                            "none",
                        ),
                        "gpu_execution_mode": model_pool["mode"],
                        "gpu_ids": ",".join(map(str, gpu_ids)),
                        "replicas": len(model_pool["contexts"]),
                        "caption": caption,
                        "inference_batch_index": batch_index,
                        "inference_batch_size": len(record_batch),
                        "batch_inference_seconds": round(
                            batch_seconds,
                            4,
                        ),
                        "frame_inference_seconds": round(
                            average_frame_seconds,
                            4,
                        ),
                        "model_load_seconds": round(load_seconds, 4),
                        **memory,
                    }
                    rows.append(row)
                    frame_rows.append(row)

                batch_fps = (
                    len(record_batch) / batch_seconds
                    if batch_seconds
                    else 0.0
                )
                frame_bar.update(len(record_batch))
                frame_bar.set_postfix(
                    batch=len(record_batch),
                    fps=f"{batch_fps:.2f}",
                    vram=f"{memory['vram_allocated_gb']:.2f}GB",
                )
                tqdm.write(
                    f"[{model_key}] batch {batch_index} "
                    f"| frames={len(record_batch)} "
                    f"| {batch_seconds:.2f}s "
                    f"| {batch_fps:.2f} frames/s"
                )

            frame_bar.close()
            peak_memory = cuda_memory_metrics(
                gpu_ids or None,
                synchronize=False,
            )
            frames_count = len(frame_rows)
            seconds_per_frame = (
                inference_seconds_total / frames_count
                if frames_count
                else 0.0
            )
            frames_per_second = (
                frames_count / inference_seconds_total
                if inference_seconds_total
                else 0.0
            )
            summary_rows.append({
                "model_key": model_pool["model_key"],
                "model_id": model_pool["contexts"][0]["model_id"],
                "backend": model_pool["contexts"][0]["backend"],
                "quantization": model_pool["contexts"][0]["spec"].get(
                    "quantization",
                    "none",
                ),
                "gpu_execution_mode": model_pool["mode"],
                "gpu_ids": ",".join(map(str, gpu_ids)),
                "replicas": len(model_pool["contexts"]),
                "per_gpu_batch_size": model_pool[
                    "per_context_batch_sizes"
                ][0],
                "global_batch_size": global_capacity,
                "frames": frames_count,
                "model_load_seconds": round(load_seconds, 4),
                "inference_seconds_total": round(
                    inference_seconds_total,
                    4,
                ),
                "seconds_per_frame": round(seconds_per_frame, 4),
                "frames_per_second": round(frames_per_second, 4),
                "total_seconds": round(
                    load_seconds + inference_seconds_total,
                    4,
                ),
                "model_loaded_vram_gb": loaded_memory[
                    "vram_allocated_gb"
                ],
                **{
                    key: value
                    for key, value in peak_memory.items()
                    if key.startswith("peak_")
                    or "_peak_" in key
                },
            })
        except Exception as exc:
            logger.exception(
                "Benchmark failed for model_key=%s: %s",
                model_key,
                exc,
            )
            spec = get_model_spec(config, model_key)
            summary_rows.append({
                "model_key": model_key,
                "model_id": spec.get("hf_id", ""),
                "backend": spec.get("backend", ""),
                "quantization": spec.get("quantization", "none"),
                "frames": len(selected),
                "error": str(exc),
            })
            if cfg_value(config, "FAIL_FAST", False):
                raise
        finally:
            unload_caption_model_pool(model_pool)

    long_df = pd.DataFrame(rows)
    summary_df = pd.DataFrame(summary_rows)

    index_columns = [
        "batch_id",
        "video_id",
        "keyframe_id",
        "frame_idx",
        "frame_sec",
        "image_gcs_uri",
        "local_image_path",
    ]
    value_columns = [
        "caption",
        "frame_inference_seconds",
        "batch_inference_seconds",
        "inference_batch_size",
        "model_load_seconds",
        "peak_vram_allocated_gb",
        "peak_vram_reserved_gb",
        "quantization",
        "gpu_execution_mode",
    ]
    if long_df.empty:
        comparison_df = pd.DataFrame()
    else:
        available_values = [
            column
            for column in value_columns
            if column in long_df.columns
        ]
        comparison_df = long_df.pivot_table(
            index=index_columns,
            columns="model_key",
            values=available_values,
            aggfunc="first",
        ).reset_index()
        comparison_df.columns = [
            "__".join(
                [str(part) for part in column if str(part)]
            )
            if isinstance(column, tuple)
            else str(column)
            for column in comparison_df.columns
        ]

    if bool(cfg_value(config, "BENCHMARK_SAVE_CSV", True)):
        long_path = layout.artifacts_dir / "benchmark_long.csv"
        comparison_path = (
            layout.artifacts_dir / "benchmark_comparison.csv"
        )
        summary_path = (
            layout.artifacts_dir / "benchmark_model_summary.csv"
        )
        long_df.to_csv(long_path, index=False)
        comparison_df.to_csv(comparison_path, index=False)
        summary_df.to_csv(summary_path, index=False)
        logger.info(
            "Saved benchmark CSV files under %s",
            layout.artifacts_dir,
        )

    return long_df, comparison_df, summary_df


def display_benchmark_outputs(
    benchmark_df: pd.DataFrame,
    image_width: int = 420,
) -> None:
    """Display each benchmark image together with its generated caption."""
    if benchmark_df is None or benchmark_df.empty:
        print("No successful benchmark captions to display.")
        return

    import base64
    import html
    from io import BytesIO

    from IPython.display import HTML, display
    from PIL import Image

    cards: list[str] = []
    sort_columns = [
        column
        for column in ["model_key", "video_id", "frame_idx"]
        if column in benchmark_df.columns
    ]
    ordered = (
        benchmark_df.sort_values(sort_columns)
        if sort_columns
        else benchmark_df
    )

    for _, row in ordered.iterrows():
        image_path = Path(str(row["local_image_path"]))
        if not image_path.exists():
            continue

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image.thumbnail((int(image_width), int(image_width)))
            buffer = BytesIO()
            image.save(buffer, format="JPEG", quality=88)
            encoded = base64.b64encode(buffer.getvalue()).decode("ascii")

        caption = html.escape(str(row.get("caption", "")))
        model_key = html.escape(str(row.get("model_key", "")))
        video_id = html.escape(str(row.get("video_id", "")))
        keyframe_id = html.escape(str(row.get("keyframe_id", "")))
        seconds = float(row.get("frame_inference_seconds", 0.0))
        vram = float(row.get("peak_vram_allocated_gb", 0.0))

        cards.append(
            f"""
            <div style="
                border:1px solid #ddd;
                border-radius:10px;
                padding:12px;
                background:white;
            ">
              <img
                src="data:image/jpeg;base64,{encoded}"
                style="max-width:100%;border-radius:6px;"
              />
              <div style="margin-top:8px;font-weight:700;">
                {model_key} · {video_id}
              </div>
              <div style="font-size:12px;color:#666;">
                {keyframe_id} · {seconds:.2f}s · peak {vram:.2f} GiB
              </div>
              <div style="margin-top:8px;line-height:1.45;">
                {caption}
              </div>
            </div>
            """
        )

    display(
        HTML(
            """
            <div style="
                display:grid;
                grid-template-columns:repeat(auto-fit,minmax(320px,1fr));
                gap:14px;
            ">
            """
            + "".join(cards)
            + "</div>"
        )
    )


def run_frame_extractor(
    config: Any,
    batches: list[str],
    max_frames: int | None,
    run_kind: str,
) -> dict[str, Any]:
    """Run captioning, store run lineage, and optionally publish canonical data."""
    import torch

    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    batch_partition = batches[0] if len(batches) == 1 else "all"
    active_model_key = str(cfg_value(config, "ACTIVE_MODEL_KEY"))
    active_spec = get_model_spec(config, active_model_key)

    layout = make_run_layout(
        config,
        batch_partition,
        run_kind,
        model_key=active_model_key,
    )
    logger = setup_logging(layout)
    write_json(layout.config_path, build_config_snapshot(config))
    started = time.perf_counter()

    records = discover_frame_records(
        config,
        bucket,
        batches,
        max_frames=max_frames,
    )
    discovered_count = len(records)
    logger.info(
        "Discovered %d frames for batches=%s",
        discovered_count,
        batches,
    )
    logger.info(
        "Run output: gs://%s/%s",
        bucket.name,
        layout.output_prefix,
    )
    logger.info(
        "Canonical data: gs://%s/%s",
        bucket.name,
        layout.data_prefix,
    )

    processed_keys = load_processed_keyframes(config, bucket, batches)
    if processed_keys:
        before_count = len(records)
        records = [
            record
            for record in records
            if record["keyframe_id"] not in processed_keys
        ]
        skipped = before_count - len(records)
        logger.info(
            "Canonical resume skipped %d already processed frames",
            skipped,
        )
    else:
        skipped = 0

    remaining_count = len(records)
    records = (
        download_frames(records, layout, client, config)
        if records
        else []
    )

    processed = 0
    failed = 0
    model_pool = None
    model_load_seconds = 0.0
    actual_execution_mode = resolve_gpu_execution_mode(config)
    replica_count = 0
    global_batch_capacity = 0

    gpu_ids = resolve_gpu_ids(config) if torch.cuda.is_available() else []
    reset_cuda_peak_memory(gpu_ids or None)
    memory = cuda_memory_metrics(gpu_ids or None)

    try:
        if records:
            load_started = time.perf_counter()
            model_pool = load_caption_model_pool(
                config,
                logger,
                active_model_key,
            )
            gpu_ids = model_pool["gpu_ids"]
            actual_execution_mode = model_pool["mode"]
            replica_count = len(model_pool["contexts"])
            global_batch_capacity = int(model_pool["global_batch_size"])
            if torch.cuda.is_available():
                synchronize_cuda(gpu_ids or None)
            model_load_seconds = time.perf_counter() - load_started
            logger.info(
                "Inference mode=%s replicas=%d gpu_ids=%s "
                "global_batch_capacity=%d",
                model_pool["mode"],
                len(model_pool["contexts"]),
                gpu_ids,
                model_pool["global_batch_size"],
            )

        batch_size = max(
            1,
            int(cfg_value(config, "PIPELINE_BATCH_SIZE", 8)),
        )
        total_batches = (
            (len(records) + batch_size - 1) // batch_size
            if records
            else 0
        )
        metrics_every = max(
            1,
            int(cfg_value(config, "METRICS_EVERY_N_BATCHES", 10)),
        )

        batches_to_run = iter_batches(records, batch_size)
        for batch_index, record_batch in enumerate(
            tqdm(
                batches_to_run,
                total=total_batches,
                desc=f"{cfg_value(config, 'EXTRACTOR_NAME')} batches",
                disable=not cfg_value(config, "USE_TQDM", True),
            ),
            start=1,
        ):
            batch_started = time.perf_counter()
            should_sample_memory = (
                batch_index == 1
                or batch_index % metrics_every == 0
                or batch_index == total_batches
            )

            try:
                output_records = extract_task_batch(
                    record_batch,
                    model_pool,
                    config,
                    layout,
                )
                append_jsonl(layout.annotations_path, output_records)
                processed += len(output_records)
                elapsed = time.perf_counter() - batch_started

                if should_sample_memory:
                    memory = cuda_memory_metrics(
                        gpu_ids or None,
                        synchronize=True,
                    )

                append_metric(
                    layout.metrics_path,
                    {
                        "run_id": layout.run_id,
                        "batch_index": batch_index,
                        "model_key": active_model_key,
                        "model_id": active_spec["hf_id"],
                        "gpu_execution_mode": (
                            model_pool["mode"]
                            if model_pool
                            else resolve_gpu_execution_mode(config)
                        ),
                        "gpu_ids": ",".join(map(str, gpu_ids)),
                        "replicas": (
                            len(model_pool["contexts"])
                            if model_pool
                            else 0
                        ),
                        "frames": len(record_batch),
                        "processed": len(output_records),
                        "failed": 0,
                        "seconds": round(elapsed, 3),
                        "frames_per_second": (
                            round(len(record_batch) / elapsed, 4)
                            if elapsed
                            else 0
                        ),
                        "memory_sampled": should_sample_memory,
                        **memory,
                    },
                )
            except Exception as exc:
                failed += len(record_batch)
                append_jsonl(
                    layout.errors_path,
                    [
                        {
                            **base_annotation(
                                record,
                                config,
                                "captioning",
                                layout.run_id,
                            ),
                            "model_key": active_model_key,
                            "model_id": active_spec["hf_id"],
                            "error": str(exc),
                        }
                        for record in record_batch
                    ],
                )

                memory = cuda_memory_metrics(
                    gpu_ids or None,
                    synchronize=True,
                )
                append_metric(
                    layout.metrics_path,
                    {
                        "run_id": layout.run_id,
                        "batch_index": batch_index,
                        "model_key": active_model_key,
                        "model_id": active_spec["hf_id"],
                        "gpu_execution_mode": (
                            model_pool["mode"]
                            if model_pool
                            else resolve_gpu_execution_mode(config)
                        ),
                        "gpu_ids": ",".join(map(str, gpu_ids)),
                        "replicas": (
                            len(model_pool["contexts"])
                            if model_pool
                            else 0
                        ),
                        "frames": len(record_batch),
                        "processed": 0,
                        "failed": len(record_batch),
                        "seconds": round(
                            time.perf_counter() - batch_started,
                            3,
                        ),
                        "frames_per_second": 0,
                        "memory_sampled": True,
                        **memory,
                    },
                )
                logger.exception(
                    "Batch %d failed: %s",
                    batch_index,
                    exc,
                )
                if (
                    "out of memory" in str(exc).lower()
                    and torch.cuda.is_available()
                ):
                    for gpu_id in gpu_ids:
                        with torch.cuda.device(gpu_id):
                            torch.cuda.empty_cache()
                if cfg_value(config, "FAIL_FAST", False):
                    raise

            total_seen = processed + failed + skipped
            if (
                total_seen
                and total_seen
                % int(cfg_value(config, "LOG_EVERY_N_FRAMES", 64))
                < batch_size
            ):
                logger.info(
                    "Progress: %d/%d processed=%d skipped=%d "
                    "failed=%d remaining=%d",
                    total_seen,
                    discovered_count,
                    processed,
                    skipped,
                    failed,
                    remaining_count,
                )
    finally:
        memory = cuda_memory_metrics(
            gpu_ids or None,
            synchronize=True,
        )
        unload_caption_model_pool(model_pool)

    success = failed == 0
    publish_canonical = bool(
        success
        and (
            (
                run_kind.startswith("full")
                and cfg_value(
                    config,
                    "PUBLISH_CANONICAL_ON_FULL",
                    True,
                )
            )
            or (
                run_kind == "demo"
                and cfg_value(
                    config,
                    "PUBLISH_CANONICAL_ON_DEMO",
                    False,
                )
            )
        )
    )

    summary = {
        "run_id": layout.run_id,
        "status": (
            "SUCCESS"
            if success
            else "COMPLETED_WITH_ERRORS"
        ),
        "extractor": cfg_value(config, "EXTRACTOR_NAME"),
        "extractor_version": cfg_value(config, "EXTRACTOR_VERSION"),
        "annotation_version": cfg_value(config, "ANNOTATION_VERSION"),
        "dataset_id": cfg_value(config, "DATASET_ID"),
        "batch_partition": batch_partition,
        "batches": batches,
        "model_key": active_model_key,
        "model_id": active_spec["hf_id"],
        "backend": active_spec["backend"],
        "quantization": active_spec.get("quantization", "none"),
        "gpu_execution_mode": actual_execution_mode,
        "gpu_ids": gpu_ids,
        "replicas": replica_count,
        "global_batch_capacity": global_batch_capacity,
        "planned_frames": discovered_count,
        "processed_frames": processed,
        "skipped_frames": skipped,
        "failed_frames": failed,
        "pipeline_batch_size": int(
            cfg_value(config, "PIPELINE_BATCH_SIZE", 8)
        ),
        "model_load_seconds": round(model_load_seconds, 3),
        "duration_seconds": round(
            time.perf_counter() - started,
            3,
        ),
        **memory,
        "run_output_prefix": (
            f"gs://{bucket.name}/{layout.output_prefix}"
        ),
        "canonical_data_prefix": (
            f"gs://{bucket.name}/{layout.data_prefix}"
        ),
        "latest_pointer_uri": (
            f"gs://{bucket.name}/{layout.latest_pointer_blob}"
        ),
        "published_to_canonical": publish_canonical,
        "created_at": utc_now(),
    }
    write_json(layout.summary_path, summary)

    if cfg_value(config, "UPLOAD_TO_GCS", True):
        if cfg_value(config, "UPLOAD_RUN_ARTIFACTS", True):
            upload_run_artifacts(bucket, layout, success)

        if publish_canonical:
            publish_manifest = publish_canonical_data(
                bucket,
                layout,
                config,
                summary,
            )
            summary["canonical_record_count"] = (
                publish_manifest["record_count"]
            )
            summary["canonical_annotations_uri"] = (
                publish_manifest["annotations_uri"]
            )
            write_json(layout.summary_path, summary)
            # Re-upload summary so the run artifact includes publish results.
            if cfg_value(config, "UPLOAD_RUN_ARTIFACTS", True):
                upload_file(
                    bucket,
                    layout.summary_path,
                    layout.output_prefix + layout.summary_path.name,
                    "application/json",
                )

    if cfg_value(
        config,
        "CLEANUP_LOCAL_FRAMES_AFTER_RUN",
        True,
    ):
        shutil.rmtree(layout.frames_dir, ignore_errors=True)

    return summary


def run_demo(config: Any) -> dict[str, Any]:
    """Run ACTIVE_MODEL_KEY on a small end-to-end sample."""
    return run_frame_extractor(
        config,
        [
            str(batch).upper()
            for batch in cfg_value(config, "DEMO_BATCHES", ["L21"])
        ],
        cfg_value(config, "DEMO_MAX_FRAMES", 32),
        "demo",
    )


def run_full(config: Any) -> list[dict[str, Any]]:
    """Run ACTIVE_MODEL_KEY over all configured batches with a safety guard."""
    if cfg_value(config, "CONFIRM_FULL_RUN", "") != "RUN_FULL_DATASET":
        print(
            'Skipped full run. Set CONFIRM_FULL_RUN = "RUN_FULL_DATASET" '
            "in Parameters and rerun that cell."
        )
        return []

    return [
        run_frame_extractor(
            config,
            [str(batch).upper()],
            cfg_value(config, "FULL_MAX_FRAMES", None),
            f"full_{str(batch).lower()}",
        )
        for batch in cfg_value(config, "BATCHES", [])
    ]


## 4. Dry Run

Kiểm tra Kaggle Secrets, GCS, manifest, model registry và output path.  
Dry Run **không tải frame và không gọi `from_pretrained()`**.


In [ ]:
dry_summary = dry_run(cfg, max_frames=cfg.DRY_RUN_MAX_FRAMES)
dry_summary


## 4b. Benchmark 32 Frames

The benchmark now sends real batches to the model and, in replicated mode,
runs one model copy on each selected GPU. Image rendering is disabled by default
so notebook display work does not distort CPU/GPU measurements.


In [ ]:
# Benchmark 32 representative frames with real batched inference.
# Default: Qwen3-VL-4B NF4, replicated across GPU 0 and GPU 1.
benchmark_long_df, benchmark_comparison_df, benchmark_summary_df = (
    benchmark_caption_models(cfg)
)

print("Model-level benchmark summary")
display(benchmark_summary_df)

print("Caption results: one row per model and frame")
display_columns = [
    "model_key",
    "video_id",
    "keyframe_id",
    "frame_idx",
    "caption",
    "gpu_execution_mode",
    "gpu_ids",
    "inference_batch_index",
    "inference_batch_size",
    "batch_inference_seconds",
    "frame_inference_seconds",
    "peak_vram_allocated_gb",
    "peak_vram_reserved_gb",
]
display(
    benchmark_long_df[
        [
            column
            for column in display_columns
            if column in benchmark_long_df.columns
        ]
    ]
    if not benchmark_long_df.empty
    else benchmark_long_df
)

if cfg.BENCHMARK_SHOW_IMAGES:
    display_benchmark_outputs(
        benchmark_long_df,
        image_width=cfg.BENCHMARK_IMAGE_WIDTH,
    )

print("Wide comparison table")
display(benchmark_comparison_df)


## 5. Demo Run

Demo chạy end-to-end với `ACTIVE_MODEL_KEY`.

- Artifact được upload vào `runs/run_id=.../`.
- Mặc định **không** publish vào canonical `data/`.
- Để thử publish canonical bằng demo, đặt `PUBLISH_CANONICAL_ON_DEMO = True`.


In [ ]:
# demo_summary = run_demo(cfg)
# demo_summary


## 6. Full Run

Full Run được khóa an toàn. Đặt:

```python
CONFIRM_FULL_RUN = "RUN_FULL_DATASET"
```

Sau khi một batch hoàn tất không có lỗi, notebook:

1. Upload lineage vào `runs/run_id=.../`.
2. Merge canonical cũ với caption mới.
3. Deduplicate theo `keyframe_id`.
4. Upload `data/annotations.jsonl` và `data/manifest.json`.
5. Ghi `data/_SUCCESS` cuối cùng.
6. Cập nhật `latest.json`.


In [ ]:
# full_summaries = run_full(cfg)
# full_summaries


## 7. Inspect Latest Local Artifacts

**Note:** Liệt kê các artifact gần nhất, bao gồm JSONL/metrics của extractor và CSV benchmark nếu đã chạy.


In [ ]:
# run_root = Path(cfg.RUN_ROOT)
# latest = sorted(
#     [path for path in run_root.glob("*") if path.is_dir()],
#     key=lambda path: path.stat().st_mtime,
#     reverse=True,
# )[:5]

# if not latest:
#     print(f"No runs found under {run_root}")

# artifact_names = [
#     "artifacts/summary.json",
#     "artifacts/config.json",
#     "artifacts/manifest.json",
#     "artifacts/published_annotations.jsonl",
#     "artifacts/annotations.jsonl",
#     "artifacts/errors.jsonl",
#     "artifacts/metrics.csv",
#     "artifacts/benchmark_long.csv",
#     "artifacts/benchmark_comparison.csv",
#     "artifacts/benchmark_model_summary.csv",
#     "run.log",
# ]

# for path in latest:
#     print(path)
#     for artifact in artifact_names:
#         candidate = path / artifact
#         if candidate.exists():
#             print("  ", candidate, candidate.stat().st_size, "bytes")
